# RL Stock Trading Agent
## EMA Crossover + Bollinger Band Squeeze Strategy (Buy Only)

This notebook implements a **Reinforcement Learning stock trading agent** that combines:

1. **EMA Crossover** (EMA 9 crosses above EMA 21 → bullish signal)
2. **Bollinger Band Squeeze** (narrowing BB width → low volatility breakout predictor)
3. **Buy-Only / Long-Only** strategy (no short selling)

The agent uses **PPO (Proximal Policy Optimization)** from Stable-Baselines3 to learn when to enter trades and which stop-loss / take-profit levels to use.

### Strategy Logic
- **Entry Signal**: EMA 9 > EMA 21 (bullish cross) **AND** BB width at 6-month low (squeeze) **AND** ADX > 15 (trending)
- **Exit**: Manual CLOSE, SL hit, or TP hit
- **Risk Control**: Dynamic SL/TP selection, max drawdown cap at 25%

### Data
- Top 25 US stocks from Tickers_list_USA.csv
- 5 years of daily OHLCV data (2021-2026)
- Total: ~28,300 rows across NVDA, AAPL, MSFT, AMZN, GOOG, META, TSLA, etc.

In [ ]:
# ============================================================
# Setup: Imports and Configuration
# ============================================================
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import CheckpointCallback

from src.stock_indicators import load_and_process_stocks, compute_stock_indicators
from src.stock_env import StockTradingEnv

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100
plt.style.use('seaborn-v0_8-darkgrid')

print('Environment ready!')

## 1. Load & Preprocess Stock Data

We load the combined stock dataset (25 symbols, ~28k rows) and compute all technical indicators.

In [ ]:
df, feature_cols = load_and_process_stocks('data/stocks/combined_stocks.parquet')

print(f'Total rows: {len(df):,}')
print(f'Symbols: {df["Symbol"].nunique()}')
print(f'Date range: {df["Date"].min().date()} to {df["Date"].max().date()}')
print(f'\nFeature columns ({len(feature_cols)}):')
for i, col in enumerate(feature_cols, 1):
    print(f'  {i:2d}. {col}')

In [ ]:
# Visualize a single stock with indicators
sample = df[df['Symbol'] == 'NVDA'].tail(250).copy()

fig, axes = plt.subplots(3, 2, figsize=(18, 14))

# Price + BB + EMAs
ax = axes[0, 0]
ax.plot(sample['Date'], sample['Close'], 'k-', linewidth=1, label='Close')
ax.fill_between(sample['Date'], sample['bb_upper'], sample['bb_lower'], alpha=0.2, color='blue', label='BB (20,2)')
ax.plot(sample['Date'], sample['ema_9'], 'b-', linewidth=0.8, alpha=0.6, label='EMA 9')
ax.plot(sample['Date'], sample['ema_21'], 'orange', linewidth=0.8, alpha=0.6, label='EMA 21')
ax.plot(sample['Date'], sample['ema_50'], 'r-', linewidth=0.8, alpha=0.6, label='EMA 50')
ax.set_title('NVDA — Price, EMAs & Bollinger Bands')
ax.legend(loc='upper left', fontsize=8)
ax.grid(alpha=0.3)

# BB Width (Squeeze detection)
ax = axes[0, 1]
ax.plot(sample['Date'], sample['bb_width'] * 100, 'b-', linewidth=1, label='BB Width (%)')
ax.axhline(y=5, color='r', linestyle='--', alpha=0.5, label='Squeeze threshold (5%)')
sqz = sample[sample['bb_squeeze'] == 1]
ax.scatter(sqz['Date'], sqz['bb_width'] * 100, color='red', s=30, alpha=0.7, marker='v', label='BB Squeeze')
ax.set_title('BB Squeeze Detection (Width < 5% AND near 6-month low)')
ax.legend(loc='upper left', fontsize=8)
ax.grid(alpha=0.3)

# EMA spread
ax = axes[1, 0]
ax.plot(sample['Date'], sample['ema_9_21_spread'], 'g-', linewidth=1, label='EMA 9-21 Spread')
ax.fill_between(sample['Date'], 0, sample['ema_9_21_spread'], where=(sample['ema_9_21_spread']>0), alpha=0.2, color='green')
ax.fill_between(sample['Date'], 0, sample['ema_9_21_spread'], where=(sample['ema_9_21_spread']<0), alpha=0.2, color='red')
cross_up = sample[sample['ema_cross_up'] == 1]
ax.scatter(cross_up['Date'], cross_up['ema_9_21_spread'], color='lime', s=50, marker='^', label='Cross Up')
ax.set_title('EMA 9-21 Spread & Cross Signals')
ax.legend(loc='upper left', fontsize=8)
ax.grid(alpha=0.3)

# RSI + ADX
ax = axes[1, 1]
ax.plot(sample['Date'], sample['rsi_14'], 'purple', linewidth=1, label='RSI (14)')
ax.plot(sample['Date'], sample['adx_14'] * 100, 'brown', linewidth=1, label='ADX (14)')
ax.axhline(y=70, color='r', linestyle='--', alpha=0.3)
ax.axhline(y=30, color='g', linestyle='--', alpha=0.3)
ax.axhline(y=15, color='brown', linestyle='--', alpha=0.3, label='ADX threshold (15)')
ax.set_title('RSI & ADX (Trend Strength)')
ax.legend(loc='upper left', fontsize=8)
ax.grid(alpha=0.3)

# Combined buy signals
ax = axes[2, 0]
ax.plot(sample['Date'], sample['Close'], 'k-', linewidth=0.8, alpha=0.5)
signals = sample[sample['combined_buy_signal'] == 1]
ax.scatter(signals['Date'], signals['Close'], color='lime', s=80, marker='^',
           edgecolors='black', linewidths=0.5, label=f'Combined BUY Signal (n={len(signals)})')
ax.set_title('Combined BUY Signals (EMA Cross Up + BB Squeeze + ADX>15)')
ax.legend(loc='upper left', fontsize=8)
ax.grid(alpha=0.3)

# Returns
ax = axes[2, 1]
ax.bar(sample['Date'], sample['returns_1d'], color=np.where(sample['returns_1d']>=0, 'green', 'red'), alpha=0.6)
ax.plot(sample['Date'], sample['returns_20d'], 'b-', linewidth=1, label='20-day return')
ax.set_title('Daily Returns & 20-Day Rolling Return')
ax.legend(loc='upper left', fontsize=8)
ax.grid(alpha=0.3)

plt.suptitle('NVDA — Technical Indicator Dashboard', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Train/Test Split

70/30 time-based split, done per-symbol to respect temporal ordering.

In [ ]:
def split_by_symbol(df, train_ratio=0.7):
    """Split data per symbol, respecting time order."""
    train_parts, test_parts = [], []
    for sym in sorted(df['Symbol'].unique()):
        sym_df = df[df['Symbol'] == sym].sort_values('Date')
        n = len(sym_df)
        split_n = int(n * train_ratio)
        train_parts.append(sym_df.iloc[:split_n])
        test_parts.append(sym_df.iloc[split_n:])
    return pd.concat(train_parts).reset_index(drop=True), pd.concat(test_parts).reset_index(drop=True)

train_df, test_df = split_by_symbol(df, train_ratio=0.7)

print(f'Train: {len(train_df):,} rows | {train_df["Symbol"].nunique()} symbols')
print(f'Test:  {len(test_df):,} rows | {test_df["Symbol"].nunique()} symbols')
print(f'Train period: {train_df["Date"].min().date()} to {train_df["Date"].max().date()}')
print(f'Test period:  {test_df["Date"].min().date()} to {test_df["Date"].max().date()}')

## 3. Feature Normalization

Z-score normalization computed from training data, applied to both train and test.

In [ ]:
feature_mean = train_df[feature_cols].values.astype(np.float32).mean(axis=0)
feature_std = train_df[feature_cols].values.astype(np.float32).std(axis=0)
feature_std[feature_std == 0] = 1.0

print(f'Feature mean range: [{feature_mean.min():.4f}, {feature_mean.max():.4f}]')
print(f'Feature std range:  [{feature_std.min():.4f}, {feature_std.max():.4f}]')

## 4. Stock Trading Environment

The `StockTradingEnv` is a **buy-only** custom Gymnasium environment with:
- **Observation**: 30-day rolling window of 22 features + 4 state features
- **Action**: HOLD (0), BUY with SL%/TP% (1-25), CLOSE (26)
- **Position persistence**: Once bought, held until CLOSE, SL hit, or TP hit
- **BB Squeeze features**: bb_width, bb_width_rank, bb_squeeze binary
- **Risk**: Drawdown-based early termination at 25%

In [ ]:
# Test the environment with random actions
SL_PCT = [2, 3, 5, 7, 10]     # Stop loss %
TP_PCT = [3, 5, 8, 12, 20]    # Take profit %

test_env = StockTradingEnv(
    df=train_df,
    window_size=30,
    sl_options_pct=SL_PCT,
    tp_options_pct=TP_PCT,
    feature_columns=feature_cols,
    feature_mean=feature_mean,
    feature_std=feature_std,
    random_start=True,
    episode_max_steps=500,
    hold_reward_weight=0.05,
    buy_penalty_pct=0.1,
    time_penalty_pct=0.005,
)

print(f'Observation shape: {test_env.observation_space.shape}')
print(f'Action space: {test_env.action_space.n} actions')
print(f'  Action 0: HOLD')
print(f'  Actions 1-25: BUY(SL%, TP%) with {len(SL_PCT)}x{len(TP_PCT)} combinations')
print(f'  Action 26: CLOSE')

In [ ]:
# Run random agent for comparison
obs, _ = test_env.reset()
random_eq = []
for _ in range(500):
    action = test_env.action_space.sample()
    obs, reward, terminated, truncated, info = test_env.step(action)
    random_eq.append(info['equity'])
    if terminated or truncated:
        break

print(f'Random agent: ${random_eq[-1]:,.2f} from $100,000 ({(random_eq[-1]/100000 - 1)*100:+.2f}%)')
print(f'Trades: {len(test_env.trade_history)}')

## 5. PPO Training

Train the agent using PPO with:
- Policy network: [256, 256, 128]
- Value network: [256, 256, 128]
- 200,000 total timesteps
- Checkpointing every 50k steps

In [ ]:
# Build training environment
def make_env():
    return StockTradingEnv(
        df=train_df, window_size=30,
        sl_options_pct=SL_PCT, tp_options_pct=TP_PCT,
        feature_columns=feature_cols,
        feature_mean=feature_mean, feature_std=feature_std,
        random_start=True, min_episode_steps=200, episode_max_steps=1500,
        hold_reward_weight=0.05, buy_penalty_pct=0.1, time_penalty_pct=0.005,
        max_drawdown_pct=0.25, drawdown_penalty_weight=2.0,
    )

train_env = DummyVecEnv([make_env])

model = PPO(
    'MlpPolicy', train_env, verbose=1,
    learning_rate=3e-4, n_steps=2048, batch_size=64, n_epochs=10,
    gamma=0.99, gae_lambda=0.95, clip_range=0.2, ent_coef=0.01,
    policy_kwargs=dict(net_arch=dict(pi=[256, 256, 128], vf=[256, 256, 128])),
)

os.makedirs('checkpoints', exist_ok=True)
ckpt = CheckpointCallback(save_freq=50_000, save_path='./checkpoints/', name_prefix='stock_ppo')

print('Training for 200,000 steps...')
model.learn(total_timesteps=200_000, callback=ckpt)
print('Training complete!')

os.makedirs('models', exist_ok=True)
model.save('models/stock_trader_best')
print('Model saved!')

## 6. Evaluation

Evaluate on both in-sample (train) and out-of-sample (test) data.

In [ ]:
def evaluate(model, eval_df):
    """Run evaluation and return equity curve + trades."""
    env = DummyVecEnv([lambda: StockTradingEnv(
        df=eval_df, window_size=30,
        sl_options_pct=SL_PCT, tp_options_pct=TP_PCT,
        feature_columns=feature_cols,
        feature_mean=feature_mean, feature_std=feature_std,
        random_start=False, episode_max_steps=None,
        hold_reward_weight=0.0, buy_penalty_pct=0.0, time_penalty_pct=0.0,
    )])
    obs = env.reset()
    eq = []
    while True:
        action, _ = model.predict(obs, deterministic=True)
        step_out = env.step(action)
        if len(step_out) == 4:
            obs, _, dones, infos = step_out
            done = bool(dones[0])
        else:
            obs, _, terminated, truncated, infos = step_out
            done = bool(terminated[0] or truncated[0])
        info = infos[0] if isinstance(infos, (list, tuple)) else infos
        eq.append(info.get('equity', env.get_attr('equity')[0]))
        if done:
            break
    return np.array(eq), env.get_attr('trade_history')[0]

print('Evaluating...')
train_eq, train_trades = evaluate(model, train_df)
test_eq, test_trades = evaluate(model, test_df)

In [ ]:
# Compute metrics
def compute_metrics(eq, trades, init=100000):
    eq = np.array(eq)
    final = eq[-1]
    ret = (final / init - 1) * 100
    r = np.diff(eq) / eq[:-1]
    r = r[~np.isnan(r) & ~np.isinf(r)]
    sharpe = np.mean(r) / np.std(r) * np.sqrt(252) if len(r)>0 and np.std(r)>0 else 0
    peak = np.maximum.accumulate(eq)
    max_dd = np.max((peak - eq) / peak) * 100
    
    n_trades = len(trades)
    if n_trades > 0:
        pnls = [t.get('net_pct', 0) for t in trades]
        wins = sum(1 for p in pnls if p > 0)
        wr = wins / n_trades * 100
    else:
        pnls = []; wins = 0; wr = 0
    
    return {
        'final': final, 'return_pct': ret, 'sharpe': sharpe, 'max_dd': max_dd,
        'n_trades': n_trades, 'win_rate': wr, 'total_pnl_pct': sum(pnls) if pnls else 0,
    }

train_m = compute_metrics(train_eq, train_trades)
test_m = compute_metrics(test_eq, test_trades)

print('='*70)
print(f'{"Metric":<25} {"Train (IS)":>18} {"Test (OOS)":>18}')
print('='*70)
print(f'{"Final Equity":<25} ${train_m["final"]:>17,.2f} ${test_m["final"]:>17,.2f}')
print(f'{"Total Return":<25} {train_m["return_pct"]:>17.2f}% {test_m["return_pct"]:>17.2f}%')
print(f'{"Sharpe Ratio":<25} {train_m["sharpe"]:>18.2f} {test_m["sharpe"]:>18.2f}')
print(f'{"Max Drawdown":<25} {train_m["max_dd"]:>17.2f}% {test_m["max_dd"]:>17.2f}%')
print(f'{"Total Trades":<25} {train_m["n_trades"]:>18} {test_m["n_trades"]:>18}')
print(f'{"Win Rate":<25} {train_m["win_rate"]:>17.1f}% {test_m["win_rate"]:>17.1f}%')
print(f'{"Net PnL %":<25} {train_m["total_pnl_pct"]:>17.2f}% {test_m["total_pnl_pct"]:>17.2f}%')
print('='*70)

## 7. Results Visualization

In [ ]:
init = 100000.0

fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# 1. Equity curves
ax = axes[0, 0]
ax.plot(train_eq, label=f'Train (IS) ${train_m["final"]:,.0f}', alpha=0.8, linewidth=1)
ax.plot(test_eq, label=f'Test (OOS) ${test_m["final"]:,.0f}', alpha=0.8, linewidth=1)
ax.axhline(y=init, color='gray', linestyle='--', alpha=0.5, label=f'Initial ${init:,.0f}')
ax.set_title('Equity Curves')
ax.legend(); ax.grid(alpha=0.3)

# 2. Drawdown (test)
ax = axes[0, 1]
peak = np.maximum.accumulate(test_eq)
dd = (peak - test_eq) / peak * 100
ax.fill_between(range(len(dd)), 0, -dd, color='red', alpha=0.3)
ax.plot(-dd, color='red', linewidth=1)
ax.set_title(f'Test Drawdown (Max: {test_m["max_dd"]:.2f}%)')
ax.grid(alpha=0.3)

# 3. Returns distribution
ax = axes[0, 2]
ret = np.diff(test_eq) / test_eq[:-1] * 100
ret = ret[~np.isnan(ret) & ~np.isinf(ret)]
ax.hist(ret, bins=50, color='steelblue', alpha=0.7, edgecolor='black')
ax.axvline(x=0, color='red', linestyle='--')
ax.set_title(f'Test Returns Distribution (Sharpe: {test_m["sharpe"]:.2f})')
ax.grid(alpha=0.3)

# 4. Trade PnL
ax = axes[1, 0]
if test_trades:
    pnls = [t.get('net_pct', 0) for t in test_trades]
    colors = ['green' if p > 0 else 'red' for p in pnls]
    ax.bar(range(len(pnls)), pnls, color=colors, alpha=0.7, edgecolor='black', linewidth=0.3)
    ax.axhline(y=0, color='black', linewidth=0.5)
    ax.set_title(f'Test Trade PnL ({len(pnls)} trades, WR: {test_m["win_rate"]:.1f}%)')
    ax.grid(alpha=0.3)

# 5. Cumulative PnL
ax = axes[1, 1]
if test_trades:
    cum = np.cumsum(pnls)
    ax.plot(cum, 'b-', linewidth=1)
    ax.fill_between(range(len(cum)), cum, 0, where=(np.array(cum)>=0), alpha=0.3, color='green')
    ax.fill_between(range(len(cum)), cum, 0, where=(np.array(cum)<0), alpha=0.3, color='red')
    ax.set_title(f'Cumulative Trade PnL ({sum(pnls):.1f}% total)')
    ax.grid(alpha=0.3)

# 6. Rolling Sharpe
ax = axes[1, 2]
if len(ret) > 20:
    roll_mean = pd.Series(ret).rolling(20).mean() * 20 * 252 / 100
    roll_std = pd.Series(ret).rolling(20).std() * np.sqrt(20 * 252) / 100
    roll_sharpe = roll_mean / roll_std
    ax.plot(roll_sharpe.values, 'b-', linewidth=1)
    ax.axhline(y=0, color='red', linestyle='--', alpha=0.5)
    ax.set_title('Rolling Sharpe (20-day)')
    ax.grid(alpha=0.3)

plt.suptitle('RL Stock Agent — EMA Crossover + BB Squeeze (Buy Only)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Benchmark: Rule-based EMA Crossover + BB Squeeze

Compare against a non-RL rule-based strategy with the same entry logic.

In [ ]:
def run_benchmark(df):
    """Rule-based EMA crossover + BB Squeeze strategy."""
    df = df.sort_values('Date').copy()
    position = 0; entry = 0; equity = 100000; eq_curve = []
    SL_PCT = 0.05; TP_PCT = 0.10
    
    for i in range(1, len(df)):
        close = df['Close'].iloc[i]
        high = df['High'].iloc[i]
        low = df['Low'].iloc[i]
        
        if position == 1:
            pnl = (close - entry) / entry
            if low <= entry * (1 - SL_PCT):
                equity *= (1 - SL_PCT - 0.002)
                position = 0
            elif high >= entry * (1 + TP_PCT):
                equity *= (1 + TP_PCT - 0.002)
                position = 0
        
        if position == 0:
            signal = (df['combined_buy_signal'].iloc[i] == 1)
            if signal:
                position = 1; entry = close
        
        eq_curve.append(equity)
    
    return np.array(eq_curve)

# Run benchmark on test set
bench_eq_curves = []
for sym in sorted(test_df['Symbol'].unique()):
    sym_df = test_df[test_df['Symbol'] == sym].sort_values('Date')
    if len(sym_df) > 50:
        bench_eq_curves.append(run_benchmark(sym_df))

if bench_eq_curves:
    # Average across symbols
    min_len = min(len(c) for c in bench_eq_curves)
    bench_avg = np.mean([c[:min_len] for c in bench_eq_curves], axis=0)
    bench_final = bench_avg[-1]
    print(f'Benchmark (rule-based EMA+BB): ${bench_final:,.2f} ({(bench_final/100000 - 1)*100:+.2f}%)')
else:
    print('No benchmark data available')

## 9. Summary & Analysis

### Strategy Recap
The RL agent attempts to learn **optimal entry timing** using:
1. **EMA Crossover** (9/21) - trend direction
2. **BB Squeeze** - low volatility breakout zones  
3. **ADX** - trend strength filter

And **optimal exit management** via dynamic SL/TP selection from 5x5=25 combinations.

### Key Challenges
- **Buy-only constraint**: In declining markets, the agent must either hold cash or take losses
- **Limited training**: 200k steps across 25 stocks is ~40 episodes per symbol
- **Market regime shifts**: 2021 bull → 2022 bear → 2023-24 recovery → 2025 bull
- **Sparse rewards**: Trades only close on SL/TP/manual close

### Improvement Roadmap
1. **Add cash/risk-free asset**: Let the agent allocate between stocks and cash
2. **Multi-stock portfolio**: Trade multiple stocks simultaneously
3. **Add sector features**: Incorporate market-wide indicators (VIX, SPY returns)
4. **Curriculum learning**: Train on easier regimes first, then harder ones
5. **Longer training**: 500k-1M steps with learning rate decay
6. **Recurrent networks (LSTM)**: Better capture temporal patterns

### Infrastructure Built
- **Pure numpy/pandas indicators** — no external TA library dependency
- **Configurable buy-only environment** — reusable for any stock universe
- **BB Squeeze detection** — relative to 6-month historical low
- **Comprehensive evaluation** — trade-level statistics and visualizations

In [ ]:
# Save final results
final = {
    'timestamp': datetime.now().isoformat(),
    'train': train_m,
    'test': test_m,
    'config': {
        'sl_pct': SL_PCT, 'tp_pct': TP_PCT,
        'symbols': sorted(test_df['Symbol'].unique().tolist()),
        'features': feature_cols,
        'strategy': 'EMA Crossover + BB Squeeze, Buy Only',
    },
}
os.makedirs('results', exist_ok=True)
with open('results/stock_final.json', 'w') as f:
    json.dump(final, f, indent=2, default=str)
print('Results saved!')
print('\nNotebook execution complete!')